# 판매자 밀집 지역 기반 마이크로 집하 허브 후보 점수화

이 노트북은 공동집하거점을 확정 설치 대상으로 판단하기보다, Olist 데이터에서 확인 가능한 주문 수, 판매자 수, 출고 준비 시간, 택배사 인계 지연율을 이용해 **우선 검토 후보 주**를 선별한다.

분석 흐름은 세 단계다.

1. 최소 물량 조건: 거점을 운영할 만큼 주문·판매자 규모가 있는지 확인한다.
2. 후보 점수 만들기: 물량 지표와 지연 지표를 백분위 점수로 변환해 통합 점수를 만든다.
3. 세 그룹 나누기: A/B/C 그룹으로 우선순위를 나눈다.

마지막에는 기준값을 바꾸었을 때 A그룹 후보가 얼마나 안정적으로 유지되는지 민감도 분석을 수행한다.

## 0. 라이브러리와 경로 설정

분석에 필요한 라이브러리를 불러오고, 정정된 item-level 데이터와 결과 저장 경로를 지정한다. 정정 데이터는 명백 오류인 중량 0과 할부 수 0이 보정되어 있고, 판매자 지오 불일치 플래그를 포함한다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:,.3f}".format)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "olist_delivered_item_level_cleaned_errors.csv"
OUTPUT_DIR = ROOT / "notebooks" / "Distribution_hub" / "SG" / "Logistics_folder"
DATA_PATH

WindowsPath('c:/team-oldest-olist-analysis/data/processed/olist_delivered_item_level_cleaned_errors.csv')

## 1. 데이터 로드와 핵심 파생 변수 생성

공동집하거점의 효과는 장거리 배송시간이 아니라 `order_approved_at` 이후 택배사 인계까지의 지연 감소로 해석한다. 따라서 `handling_time_days`와 `handover_late`를 핵심 변수로 만든다.

In [2]:
DATE_COLS = [
    "shipping_limit_date",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

df = pd.read_csv(DATA_PATH, parse_dates=DATE_COLS)
df = df[df["order_status"].eq("delivered")].copy()

df["handling_time_days"] = (
    df["order_delivered_carrier_date"] - df["order_approved_at"]
).dt.total_seconds() / 86400

df["handover_delay_days"] = (
    df["order_delivered_carrier_date"] - df["shipping_limit_date"]
).dt.total_seconds() / 86400

df["handover_late"] = df["handover_delay_days"] > 0

valid = df[
    df["seller_state"].notna()
    & df["seller_id"].notna()
    & df["order_id"].notna()
    & df["shipping_limit_date"].notna()
    & df["order_approved_at"].notna()
    & df["order_delivered_carrier_date"].notna()
    & df["handling_time_days"].notna()
    & (df["handling_time_days"] >= 0)
].copy()

print(f"원본 delivered item 수: {len(df):,}")
print(f"분석 유효 item 수: {len(valid):,}")

valid[
    ["seller_state", "seller_city", "seller_id", "order_id", "handling_time_days", "handover_delay_days", "handover_late"]
].head()

원본 delivered item 수: 108,581
분석 유효 item 수: 108,581


,seller_state,seller_city,seller_id,order_id,handling_time_days,handover_delay_days,handover_late
0,SP,volta redonda,48436dade18ac8b2bce089ec2a041202,00010242fe8c5a6d1ba2dd792cb16214,6.367,0.367,True
1,SP,sao paulo,dd7ddc04e1b6c2c614352b383efe2d36,00018f77f2f0320c557190d7a144bdd3,8.146,1.146,True
2,MG,borda da mata,5b51032eddd242adc84c38acab88f23d,000229ec398224ef6ca0657da4fc703e,1.909,-2.091,False
3,SP,franca,9d7a1d34a5052409006425275ba1c2b4,00024acbcdf0a6daa1e931b038114c75,2.137,-4.863,False
4,PR,loanda,df560393f3a51e74553ab94004ba5c87,00042b26cf59d7ce69dfabb4e55b4fd9,11.817,2.825,True


## 2. 주별 집계 테이블 만들기

주별 주문 수, 판매자 수, 판매자 도시 수, 출고 준비 시간, 택배사 인계 지연율을 집계한다. 이 단계의 결과가 이후 최소 물량 조건과 후보 점수 산정의 기본 테이블이 된다.

In [3]:
state_summary = (
    valid.groupby("seller_state")
    .agg(
        seller_region=("seller_region", "first"),
        order_items=("order_id", "size"),
        orders=("order_id", "nunique"),
        sellers=("seller_id", "nunique"),
        seller_cities=("seller_city", "nunique"),
        handling_mean_days=("handling_time_days", "mean"),
        handling_median_days=("handling_time_days", "median"),
        handling_p75_days=("handling_time_days", lambda s: s.quantile(0.75)),
        handover_late_rate=("handover_late", "mean"),
        handover_delay_mean_days=("handover_delay_days", "mean"),
        review_score_mean=("review_score_mean", "mean"),
    )
    .reset_index()
)

state_summary.sort_values("orders", ascending=False).head(10)

,seller_state,seller_region,order_items,orders,sellers,seller_cities,handling_mean_days,handling_median_days,handling_p75_days,handover_late_rate,handover_delay_mean_days,review_score_mean
21,SP,남동부,77456,67649,1760,250,2.903,1.894,3.714,0.094,-3.268,4.051
7,MG,남동부,8472,7616,235,81,2.888,1.735,3.399,0.080,-3.845,4.164
14,PR,남부,8371,7411,335,65,3.144,1.916,3.767,0.112,-3.239,4.125
15,RJ,남동부,4622,4174,163,37,2.535,1.730,3.170,0.087,-3.260,4.166
19,SC,남부,3930,3541,184,64,2.718,1.724,3.480,0.086,-3.371,4.132
18,RS,남부,2134,1929,125,51,3.113,1.938,4.014,0.092,-3.374,4.257
3,DF,중서부,871,796,30,3,2.659,1.657,3.221,0.095,-3.481,4.060
1,BA,북동부,620,546,18,12,2.954,2.204,4.094,0.073,-2.934,4.156
5,GO,중서부,502,445,39,12,2.284,1.399,2.923,0.070,-3.211,4.305
12,PE,북동부,438,396,9,4,1.812,1.287,2.635,0.041,-3.342,4.147


## 3. 1단계: 최소 물량 조건

Olist에는 거점 임대료, 인건비, 차량비 같은 비용 정보가 없으므로 손익분기점을 직접 계산할 수 없다. 대신 주문 수와 판매자 수를 **거점 운영 여건을 판단하기 위한 간접 지표**로 사용한다.

여기서는 주문 수와 판매자 수의 백분위 평균인 `volume_score`가 상위 30% 수준에 해당하는 주를 최소 물량 조건 통과로 본다. 이 조건은 확정 설치 기준이 아니라, 표본 수가 너무 작은 주의 지연 평균이 과대해석되는 것을 줄이기 위한 선별 장치다.

In [4]:
for col in ["orders", "sellers", "handling_median_days", "handover_late_rate"]:
    state_summary[f"{col}_pct"] = state_summary[col].rank(pct=True, method="average")

MIN_VOLUME_PCT_CUTOFF = 0.70

state_summary["volume_score"] = (
    0.5 * state_summary["orders_pct"]
    + 0.5 * state_summary["sellers_pct"]
)

state_summary["passes_min_volume_condition"] = state_summary["volume_score"].ge(
    MIN_VOLUME_PCT_CUTOFF
)

state_summary[
    [
        "seller_state",
        "seller_region",
        "orders",
        "sellers",
        "orders_pct",
        "sellers_pct",
        "volume_score",
        "passes_min_volume_condition",
    ]
].sort_values(["passes_min_volume_condition", "orders"], ascending=[False, False])

,seller_state,seller_region,orders,sellers,orders_pct,sellers_pct,volume_score,passes_min_volume_condition
21,SP,남동부,67649,1760,1.000,1.000,1.000,True
7,MG,남동부,7616,235,0.955,0.909,0.932,True
14,PR,남부,7411,335,0.909,0.955,0.932,True
15,RJ,남동부,4174,163,0.864,0.818,0.841,True
19,SC,남부,3541,184,0.818,0.864,0.841,True
18,RS,남부,1929,125,0.773,0.773,0.773,True
3,DF,중서부,796,30,0.727,0.682,0.705,True
1,BA,북동부,546,18,0.682,0.591,0.636,False
5,GO,중서부,445,39,0.636,0.727,0.682,False
12,PE,북동부,396,9,0.591,0.500,0.545,False


## 4. 2단계: 후보 점수 만들기

후보 점수는 물량 지표와 지연 지표를 함께 반영한다. 절대값을 그대로 쓰면 규모가 큰 변수의 영향이 과도해질 수 있으므로, 각 지표를 주별 백분위 점수로 바꾼 뒤 가중합한다.

가중치는 물량 50%, 지연 50% 구조를 유지하되, 지연 중에서는 `handling_median_days`를 조금 더 크게 반영한다.

In [5]:
state_summary["delay_score"] = (
    0.5 * state_summary["handling_median_days_pct"]
    + 0.5 * state_summary["handover_late_rate_pct"]
)

state_summary["candidate_score"] = (
    0.25 * state_summary["orders_pct"]
    + 0.25 * state_summary["sellers_pct"]
    + 0.30 * state_summary["handling_median_days_pct"]
    + 0.20 * state_summary["handover_late_rate_pct"]
)

state_summary.sort_values("candidate_score", ascending=False)[
    [
        "seller_state",
        "seller_region",
        "orders",
        "sellers",
        "handling_median_days",
        "handover_late_rate",
        "volume_score",
        "delay_score",
        "candidate_score",
    ]
].head(10)

,seller_state,seller_region,orders,sellers,handling_median_days,handover_late_rate,volume_score,delay_score,candidate_score
21,SP,남동부,67649,1760,1.894,0.094,1.000,0.545,0.773
14,PR,남부,7411,335,1.916,0.112,0.932,0.614,0.770
18,RS,남부,1929,125,1.938,0.092,0.773,0.591,0.691
7,MG,남동부,7616,235,1.735,0.080,0.932,0.409,0.675
15,RJ,남동부,4174,163,1.730,0.087,0.841,0.432,0.634
6,MA,북동부,370,1,2.412,0.332,0.330,0.932,0.628
1,BA,북동부,546,18,2.204,0.073,0.636,0.545,0.614
19,SC,남부,3541,184,1.724,0.086,0.841,0.386,0.611
8,MS,중서부,48,5,2.473,0.143,0.352,0.841,0.608
9,MT,중서부,135,4,2.221,0.133,0.386,0.750,0.575


## 5. 3단계: A/B/C 그룹 나누기

A그룹은 최소 물량 조건을 통과하면서 지연 필요성도 높은 주다. 기본 기준은 `volume_score >= 0.70`, `delay_score >= 0.60`이다. 물량 기준은 고정비가 있는 거점 운영 특성을 고려해 상위 30% 수준으로 엄격하게 두고, 지연 기준은 물량이 충분한 후보를 과도하게 배제하지 않기 위해 상위 40% 수준으로 설정했다.

B그룹은 물량과 지연 중 하나만 높은 주이며, C그룹은 둘 다 낮아 현 데이터 기준 거점 설치 우선순위가 낮은 주다.

In [6]:
DELAY_SCORE_CUTOFF = 0.60

state_summary["has_high_delay_need"] = state_summary["delay_score"].ge(DELAY_SCORE_CUTOFF)

def assign_candidate_group(row):
    if row["passes_min_volume_condition"] and row["has_high_delay_need"]:
        return "A_우선_검토"
    if row["passes_min_volume_condition"] or row["has_high_delay_need"]:
        return "B_모니터링"
    return "C_우선순위_낮음"

state_summary["candidate_group"] = state_summary.apply(assign_candidate_group, axis=1)

grouped = state_summary.sort_values(
    ["candidate_group", "candidate_score"], ascending=[True, False]
)

grouped[
    [
        "candidate_group",
        "seller_state",
        "seller_region",
        "orders",
        "sellers",
        "handling_median_days",
        "handling_p75_days",
        "handover_late_rate",
        "volume_score",
        "delay_score",
        "candidate_score",
    ]
]

,candidate_group,seller_state,seller_region,orders,sellers,handling_median_days,handling_p75_days,handover_late_rate,volume_score,delay_score,candidate_score
14,A_우선_검토,PR,남부,7411,335,1.916,3.767,0.112,0.932,0.614,0.770
21,B_모니터링,SP,남동부,67649,1760,1.894,3.714,0.094,1.000,0.545,0.773
18,B_모니터링,RS,남부,1929,125,1.938,4.014,0.092,0.773,0.591,0.691
7,B_모니터링,MG,남동부,7616,235,1.735,3.399,0.080,0.932,0.409,0.675
15,B_모니터링,RJ,남동부,4174,163,1.730,3.170,0.087,0.841,0.432,0.634
6,B_모니터링,MA,북동부,370,1,2.412,7.437,0.332,0.330,0.932,0.628
19,B_모니터링,SC,남부,3541,184,1.724,3.480,0.086,0.841,0.386,0.611
8,B_모니터링,MS,중서부,48,5,2.473,5.708,0.143,0.352,0.841,0.608
9,B_모니터링,MT,중서부,135,4,2.221,4.151,0.133,0.386,0.750,0.575
3,B_모니터링,DF,중서부,796,30,1.657,3.221,0.095,0.705,0.455,0.566


## 6. 그룹별 요약

각 그룹이 어떤 특성을 갖는지 평균값으로 비교한다. A그룹은 확정 설치 지역이 아니라, 추가 비용·입지·운영 검토를 우선적으로 진행할 후보라는 점을 유지한다.

In [7]:
group_summary = (
    state_summary.groupby("candidate_group")
    .agg(
        states=("seller_state", "nunique"),
        orders=("orders", "sum"),
        sellers=("sellers", "sum"),
        avg_handling_median_days=("handling_median_days", "mean"),
        avg_handover_late_rate=("handover_late_rate", "mean"),
        avg_volume_score=("volume_score", "mean"),
        avg_delay_score=("delay_score", "mean"),
        avg_candidate_score=("candidate_score", "mean"),
    )
    .reset_index()
)

group_summary

,candidate_group,states,orders,sellers,avg_handling_median_days,avg_handover_late_rate,avg_volume_score,avg_delay_score,avg_candidate_score
0,A_우선_검토,1,7411,335,1.916,0.112,0.932,0.614,0.770
1,B_모니터링,13,86354,2520,2.126,0.165,0.545,0.666,0.604
2,C_우선순위_낮음,8,1816,104,1.473,0.060,0.436,0.278,0.359


## 7. 민감도 분석: 기준값을 바꾸어도 후보가 유지되는가

최소 물량 조건과 지연 필요성 기준을 하나로 고정하면 임의성이 생긴다. 따라서 상위 30%, 40%, 50% 수준 및 물량/지연 기준을 서로 다르게 둔 시나리오를 비교한다.

여러 시나리오에서 반복적으로 A그룹에 들어오는 주는 기준 선택에 덜 민감한 강한 후보로 해석할 수 있다.

In [8]:
SCENARIOS = [
    ("strict_top_30", 0.70, 0.70),
    ("balanced_top_40", 0.60, 0.60),
    ("loose_top_50", 0.50, 0.50),
    ("volume_strict_delay_loose", 0.70, 0.60),
    ("volume_loose_delay_strict", 0.60, 0.70),
]

sensitivity_frames = []

for scenario, volume_threshold, delay_threshold in SCENARIOS:
    tmp = state_summary.copy()
    tmp["scenario"] = scenario
    tmp["volume_threshold"] = volume_threshold
    tmp["delay_threshold"] = delay_threshold
    tmp["passes_volume"] = tmp["volume_score"].ge(volume_threshold)
    tmp["passes_delay"] = tmp["delay_score"].ge(delay_threshold)
    tmp["scenario_group"] = np.select(
        [
            tmp["passes_volume"] & tmp["passes_delay"],
            tmp["passes_volume"] | tmp["passes_delay"],
        ],
        ["A_우선_검토", "B_모니터링"],
        default="C_우선순위_낮음",
    )
    sensitivity_frames.append(tmp)

sensitivity = pd.concat(sensitivity_frames, ignore_index=True)

sensitivity_summary = (
    sensitivity.assign(is_a=lambda d: d["scenario_group"].eq("A_우선_검토"))
    .groupby("seller_state", as_index=False)
    .agg(
        a_group_count=("is_a", "sum"),
        scenario_count=("scenario", "nunique"),
        avg_candidate_score=("candidate_score", "mean"),
        avg_volume_score=("volume_score", "mean"),
        avg_delay_score=("delay_score", "mean"),
    )
)
sensitivity_summary["a_group_stability"] = (
    sensitivity_summary["a_group_count"] / sensitivity_summary["scenario_count"]
)

sensitivity_summary.sort_values(
    ["a_group_stability", "avg_candidate_score"], ascending=[False, False]
)

,seller_state,a_group_count,scenario_count,avg_candidate_score,avg_volume_score,avg_delay_score,a_group_stability
14,PR,3,5,0.770,0.932,0.614,0.600
21,SP,1,5,0.773,1.000,0.545,0.200
18,RS,1,5,0.691,0.773,0.591,0.200
1,BA,1,5,0.614,0.636,0.545,0.200
7,MG,0,5,0.675,0.932,0.409,0.000
15,RJ,0,5,0.634,0.841,0.432,0.000
6,MA,0,5,0.628,0.330,0.932,0.000
19,SC,0,5,0.611,0.841,0.386,0.000
8,MS,0,5,0.608,0.352,0.841,0.000
9,MT,0,5,0.575,0.386,0.750,0.000


## 8. 시나리오별 A그룹 후보 확인

각 시나리오에서 A그룹으로 분류된 주를 확인한다. 특정 주가 여러 기준에서 반복적으로 등장한다면, 공동집하거점 후보로 제안할 때 더 안정적인 근거가 된다.

In [9]:
sensitivity_a = sensitivity[sensitivity["scenario_group"].eq("A_우선_검토")]

(
    sensitivity_a.groupby("scenario")["seller_state"]
    .apply(lambda s: ", ".join(sorted(s)))
    .reset_index(name="A_group_states")
)

,scenario,A_group_states
0,balanced_top_40,PR
1,loose_top_50,"BA, PR, RS, SP"
2,volume_strict_delay_loose,PR


## 9. 최종 결과 테이블 확인

주별 후보 점수표와 민감도 분석 결과를 최종 확인한다. 발표나 보고서에서는 A그룹을 확정 설치 지역이 아니라 `우선 검토 후보`로 표현한다.

In [10]:
display(
    state_summary.sort_values("candidate_score", ascending=False)[
        [
            "seller_state",
            "seller_region",
            "candidate_group",
            "orders",
            "sellers",
            "handling_median_days",
            "handover_late_rate",
            "volume_score",
            "delay_score",
            "candidate_score",
        ]
    ]
)

display(
    sensitivity_summary.sort_values(
        ["a_group_stability", "avg_candidate_score"], ascending=[False, False]
    )
)

,seller_state,seller_region,candidate_group,orders,sellers,handling_median_days,handover_late_rate,volume_score,delay_score,candidate_score
21,SP,남동부,B_모니터링,67649,1760,1.894,0.094,1.000,0.545,0.773
14,PR,남부,A_우선_검토,7411,335,1.916,0.112,0.932,0.614,0.770
18,RS,남부,B_모니터링,1929,125,1.938,0.092,0.773,0.591,0.691
7,MG,남동부,B_모니터링,7616,235,1.735,0.080,0.932,0.409,0.675
15,RJ,남동부,B_모니터링,4174,163,1.730,0.087,0.841,0.432,0.634
6,MA,북동부,B_모니터링,370,1,2.412,0.332,0.330,0.932,0.628
1,BA,북동부,C_우선순위_낮음,546,18,2.204,0.073,0.636,0.545,0.614
19,SC,남부,B_모니터링,3541,184,1.724,0.086,0.841,0.386,0.611
8,MS,중서부,B_모니터링,48,5,2.473,0.143,0.352,0.841,0.608
9,MT,중서부,B_모니터링,135,4,2.221,0.133,0.386,0.750,0.575


,seller_state,a_group_count,scenario_count,avg_candidate_score,avg_volume_score,avg_delay_score,a_group_stability
14,PR,3,5,0.770,0.932,0.614,0.600
21,SP,1,5,0.773,1.000,0.545,0.200
18,RS,1,5,0.691,0.773,0.591,0.200
1,BA,1,5,0.614,0.636,0.545,0.200
7,MG,0,5,0.675,0.932,0.409,0.000
15,RJ,0,5,0.634,0.841,0.432,0.000
6,MA,0,5,0.628,0.330,0.932,0.000
19,SC,0,5,0.611,0.841,0.386,0.000
8,MS,0,5,0.608,0.352,0.841,0.000
9,MT,0,5,0.575,0.386,0.750,0.000


## 10. 발표용 해석 문장

아래 문장은 분석 결과를 설명할 때 사용할 수 있는 요약 문장이다.

In [11]:
print(
    "공동집하거점은 모든 주에 일괄 설치하는 인프라가 아니라, "
    "주문 수와 판매자 수가 충분하면서 결제 승인 이후 택배사 인계까지의 지연이 반복적으로 나타나는 주에 "
    "우선 검토해야 하는 선택적 솔루션이다. "
    "본 분석은 실제 운영비 자료가 없다는 한계를 고려해 주문 수와 판매자 수를 운영 여건의 간접 지표로 사용하고, "
    "handling_time과 shipping_limit_date 초과 인계율을 개선 필요성 지표로 사용하였다."
)

공동집하거점은 모든 주에 일괄 설치하는 인프라가 아니라, 주문 수와 판매자 수가 충분하면서 결제 승인 이후 택배사 인계까지의 지연이 반복적으로 나타나는 주에 우선 검토해야 하는 선택적 솔루션이다. 본 분석은 실제 운영비 자료가 없다는 한계를 고려해 주문 수와 판매자 수를 운영 여건의 간접 지표로 사용하고, handling_time과 shipping_limit_date 초과 인계율을 개선 필요성 지표로 사용하였다.
